# Phase 7: SchNet 3D+2D Hybrid Experiment (Kaggle GPU)

Compare SchNet 3D-only (Phase 6 baseline R²=0.882) vs SchNet 3D + 212 RDKit 2D descriptors.

## Setup
1. Upload `pyg_3d_graphs_etkdg_expanded_with_desc.pt` (0.16 GB) to Kaggle dataset
2. Enable GPU (T4)
3. Run all cells

In [ ]:
!pip install -q torch-geometric

In [ ]:
import time, json, os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch_geometric.loader import DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print(f"torch {torch.__version__}, CUDA {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
# ── CONFIG ──
GRAPH_PATH = "/kaggle/input/molgap-hybrid/pyg_3d_graphs_etkdg_expanded_with_desc.pt"
OUTPUT_DIR = "/kaggle/working"

SEED = 42
TARGET_COLS = ["homo", "lumo", "gap"]
EPOCHS = 300
PATIENCE = 40
CHECKPOINT_EVERY = 1

torch.manual_seed(SEED)
np.random.seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# ── SchNet with 2D descriptor fusion ──
class SchNetWrapper(nn.Module):
    def __init__(self, hidden_channels, num_filters, num_interactions,
                 num_gaussians, cutoff, dropout=0.1, n_targets=3,
                 use_charges=False, n_desc=0):
        super().__init__()
        from torch_geometric.nn.models import SchNet
        self.schnet = SchNet(
            hidden_channels=hidden_channels, num_filters=num_filters,
            num_interactions=num_interactions, num_gaussians=num_gaussians,
            cutoff=cutoff,
        )
        self.use_charges = use_charges
        if use_charges:
            self.charge_proj = nn.Linear(1, hidden_channels)
        self.n_desc = n_desc
        if n_desc > 0:
            self.desc_proj = nn.Sequential(
                nn.Linear(n_desc, hidden_channels // 2),
                nn.SiLU(),
                nn.Dropout(dropout),
            )
            head_in = hidden_channels + hidden_channels // 2
        else:
            head_in = hidden_channels
        self.head = nn.Sequential(
            nn.Linear(head_in, hidden_channels),
            nn.SiLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_channels, hidden_channels // 2),
            nn.SiLU(),
            nn.Linear(hidden_channels // 2, n_targets),
        )

    def forward(self, z, pos, batch, charges=None, desc=None):
        from torch_geometric.nn import global_mean_pool
        from torch_geometric.nn.models.schnet import radius_graph
        h = self.schnet.embedding(z)
        if self.use_charges and charges is not None:
            h = h + self.charge_proj(charges.unsqueeze(-1))
        edge_index = radius_graph(pos, r=self.schnet.cutoff, batch=batch, max_num_neighbors=32)
        row, col = edge_index
        edge_weight = (pos[row] - pos[col]).norm(dim=-1)
        edge_attr = self.schnet.distance_expansion(edge_weight)
        for interaction in self.schnet.interactions:
            h = h + interaction(h, edge_index, edge_weight, edge_attr)
        h = global_mean_pool(h, batch)
        if self.n_desc > 0 and desc is not None:
            desc = desc.view(h.size(0), self.n_desc)
            h = torch.cat([h, self.desc_proj(desc)], dim=-1)
        return self.head(h)

In [ ]:
# ── Load data ──
print("Loading graphs with 2D descriptors...")
data_list = torch.load(GRAPH_PATH, weights_only=False)
print(f"Loaded {len(data_list)} graphs")

has_charges = hasattr(data_list[0], 'charges')
has_desc = hasattr(data_list[0], 'desc')
n_desc = data_list[0].desc.shape[0] if has_desc else 0
print(f"Charges: {has_charges}, 2D desc: {has_desc} ({n_desc} features)")

# Split
all_idx = np.arange(len(data_list))
train_valid_idx, test_idx = train_test_split(all_idx, test_size=0.1, random_state=SEED)
train_idx, valid_idx = train_test_split(train_valid_idx, test_size=0.1/0.9, random_state=SEED)

train_data = [data_list[i] for i in train_idx]
valid_data = [data_list[i] for i in valid_idx]
test_data  = [data_list[i] for i in test_idx]

train_y = np.stack([d.y.squeeze(0).numpy() for d in train_data])
y_mean = train_y.mean(axis=0)
y_std = train_y.std(axis=0)
y_std[y_std < 1e-6] = 1.0

for d in train_data + valid_data + test_data:
    d.y = (d.y - torch.tensor(y_mean)) / torch.tensor(y_std)

del data_list
n_total = len(train_data) + len(valid_data) + len(test_data)
print(f"Split: train={len(train_data)}, valid={len(valid_data)}, test={len(test_data)}")
print(f"y_mean={y_mean}, y_std={y_std}")

In [ ]:
# ── Profile: pick best batch size ──
print("Profiling batch sizes...")
test_params = {
    "hidden_channels": 192, "num_filters": 256, "num_interactions": 6,
    "num_gaussians": 100, "cutoff": 5.0, "dropout": 0.1,
}

for bs in [128, 256, 512]:
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    loader = DataLoader(train_data, batch_size=bs, shuffle=True)
    model = SchNetWrapper(**test_params, use_charges=has_charges, n_desc=n_desc).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
    scaler = torch.amp.GradScaler("cuda")
    model.train()
    t0 = time.time()
    try:
        for i, batch in enumerate(loader):
            if i >= 5: break
            batch = batch.to(device)
            optimizer.zero_grad()
            with torch.amp.autocast("cuda"):
                desc = getattr(batch, 'desc', None)
                out = model(batch.z, batch.pos, batch.batch,
                            charges=getattr(batch, 'charges', None), desc=desc)
                loss = F.l1_loss(out, batch.y)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        elapsed = (time.time() - t0) / 5
        n_batches = len(train_data) // bs
        peak = torch.cuda.max_memory_allocated() / 1e9
        print(f"  bs={bs:3d} | {elapsed:.2f}s/batch | ~{elapsed*n_batches:.0f}s/epoch | peak={peak:.2f}GB")
    except RuntimeError as e:
        if "out of memory" in str(e):
            print(f"  bs={bs:3d} | OOM!")
            torch.cuda.empty_cache()
        else:
            raise
    del model, optimizer, scaler
    torch.cuda.empty_cache()

In [ ]:
# ── Train hybrid model ──
# Adjust BS based on profiling above
BS = 256
PARAMS = {
    "hidden_channels": 192, "num_filters": 256, "num_interactions": 6,
    "num_gaussians": 100, "cutoff": 5.0, "dropout": 0.1,
    "lr": 1.56e-3, "weight_decay": 2.06e-5,
}

train_loader = DataLoader(train_data, batch_size=BS, shuffle=True)
valid_loader = DataLoader(valid_data, batch_size=BS)
test_loader  = DataLoader(test_data, batch_size=BS)

model = SchNetWrapper(
    **PARAMS, use_charges=has_charges, n_desc=n_desc,
).to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f"Model: {n_params:,} params, n_desc={n_desc}, cutoff={PARAMS['cutoff']}, bs={BS}")

optimizer = torch.optim.AdamW(model.parameters(),
                              lr=PARAMS["lr"], weight_decay=PARAMS["weight_decay"])
scheduler = ReduceLROnPlateau(optimizer, mode="min", factor=0.5,
                              patience=max(5, PATIENCE // 3), min_lr=1e-6)
scaler = torch.amp.GradScaler("cuda")

best_val_mae = float("inf")
best_epoch = 0
best_state = None
log_rows = []

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    # Train
    model.train()
    total_loss, n = 0, 0
    for batch in train_loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        with torch.amp.autocast("cuda"):
            out = model(batch.z, batch.pos, batch.batch,
                        charges=getattr(batch, 'charges', None),
                        desc=getattr(batch, 'desc', None))
            loss = F.l1_loss(out, batch.y)
        scaler.scale(loss).backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 10.0)
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item() * batch.num_graphs
        n += batch.num_graphs
    train_loss = total_loss / n

    # Validate
    model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for batch in valid_loader:
            batch = batch.to(device)
            with torch.amp.autocast("cuda"):
                out = model(batch.z, batch.pos, batch.batch,
                            charges=getattr(batch, 'charges', None),
                            desc=getattr(batch, 'desc', None))
            preds.append(out.cpu().numpy())
            trues.append(batch.y.cpu().numpy())
    val_pred = np.vstack(preds) * y_std + y_mean
    val_true = np.vstack(trues) * y_std + y_mean
    val_mae = float(np.mean(np.abs(val_pred - val_true)))

    scheduler.step(val_mae)
    elapsed = time.time() - t0
    log_rows.append({"epoch": epoch, "train_loss": train_loss,
                     "val_mae": val_mae, "lr": optimizer.param_groups[0]["lr"],
                     "time_s": elapsed})

    is_best = val_mae < best_val_mae
    if is_best:
        best_val_mae = val_mae
        best_epoch = epoch
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        torch.save(best_state, f"{OUTPUT_DIR}/best_model.pt")

    # Checkpoint every epoch
    if epoch % CHECKPOINT_EVERY == 0:
        torch.save({
            "epoch": epoch,
            "model_state_dict": {k: v.cpu().clone() for k, v in model.state_dict().items()},
            "optimizer_state_dict": optimizer.state_dict(),
            "best_val_mae": best_val_mae,
            "best_epoch": best_epoch,
            "y_mean": y_mean.tolist(),
            "y_std": y_std.tolist(),
            "log_rows": log_rows,
        }, f"{OUTPUT_DIR}/checkpoint.pt")

    if epoch % 10 == 0 or epoch == 1 or is_best:
        marker = " *" if is_best else ""
        print(f"  Epoch {epoch:3d} | loss={train_loss:.4f} | val_MAE={val_mae:.4f} | "
              f"best={best_val_mae:.4f}@{best_epoch} | {elapsed:.1f}s{marker}")

    if epoch - best_epoch >= PATIENCE:
        print(f"  Early stop at epoch {epoch} (best={best_epoch})")
        break

pd.DataFrame(log_rows).to_csv(f"{OUTPUT_DIR}/hybrid_train_log.csv", index=False)

In [ ]:
# ── Evaluate on test set ──
model.load_state_dict({k: v.to(device) for k, v in best_state.items()})
model.eval()
preds, trues = [], []
with torch.no_grad():
    for batch in test_loader:
        batch = batch.to(device)
        with torch.amp.autocast("cuda"):
            out = model(batch.z, batch.pos, batch.batch,
                        charges=getattr(batch, 'charges', None),
                        desc=getattr(batch, 'desc', None))
        preds.append(out.cpu().numpy())
        trues.append(batch.y.cpu().numpy())
test_pred = np.vstack(preds) * y_std + y_mean
test_true = np.vstack(trues) * y_std + y_mean

metrics = {}
print(f"\n{'='*60}")
print(f"  Hybrid 3D+2D Test Results ({n_total} molecules)")
print(f"{'='*60}")
for i, t in enumerate(TARGET_COLS):
    mae = mean_absolute_error(test_true[:, i], test_pred[:, i])
    rmse = np.sqrt(mean_squared_error(test_true[:, i], test_pred[:, i]))
    r2 = r2_score(test_true[:, i], test_pred[:, i])
    metrics[t] = {"mae": float(mae), "rmse": float(rmse), "r2": float(r2)}
    print(f"  {t:5s}: MAE={mae:.4f}  RMSE={rmse:.4f}  R2={r2:.4f}")

avg = {k: np.mean([m[k] for m in metrics.values()]) for k in ["mae", "rmse", "r2"]}
metrics["average"] = {k: float(v) for k, v in avg.items()}
print(f"  avg  : MAE={avg['mae']:.4f}  RMSE={avg['rmse']:.4f}  R2={avg['r2']:.4f}")

# Phase 6 baseline
p6_mae, p6_r2 = 0.1620, 0.8823
print(f"\n  Phase 6 (3D-only):  avg MAE={p6_mae:.4f}  R2={p6_r2:.4f}")
print(f"  Hybrid (3D+2D):    avg MAE={avg['mae']:.4f}  R2={avg['r2']:.4f}")
print(f"  Delta: R2 {avg['r2']-p6_r2:+.4f}, MAE {p6_mae-avg['mae']:+.4f}")

In [ ]:
# ── Save outputs ──
torch.save(best_state, f"{OUTPUT_DIR}/gnn_schnet_3d2d_hybrid.pt")

with open(f"{OUTPUT_DIR}/hybrid_comparison.json", "w") as f:
    json.dump({
        "experiment": "hybrid_2d3d",
        "n_data": n_total,
        "n_desc": n_desc,
        "params": PARAMS,
        "batch_size": BS,
        "n_params": n_params,
        "best_epoch": best_epoch,
        "metrics": metrics,
        "phase6_baseline": {"avg_mae": p6_mae, "avg_r2": p6_r2},
        "delta_avg_r2": float(avg['r2'] - p6_r2),
        "delta_avg_mae": float(p6_mae - avg['mae']),
        "y_mean": y_mean.tolist(),
        "y_std": y_std.tolist(),
    }, f, indent=2)

print(f"\nSaved to {OUTPUT_DIR}/")
print(f"  gnn_schnet_3d2d_hybrid.pt")
print(f"  hybrid_comparison.json")
print(f"  hybrid_train_log.csv")
print(f"  best_model.pt / checkpoint.pt")
print(f"\nDownload and place locally:")
print(f"  .pt   -> models/")
print(f"  .json -> results/phase7/hybrid_2d3d/")